# Spike 0.6c — Tier-1 unsteady-cfg + NACA 0012 benchmark on Colab

**Spec reference:** `docs/plan_R11.md §Phase 0 Spike 0.6c` (Round-9 HIGH-10 / H10 lock).

**This notebook is Colab-only.** It imports `google.colab` unconditionally at the top. If you run it elsewhere (local Jupyter, VS Code, etc.) cell 1 will fail with `ImportError: No module named 'google.colab'` — that's correct; this notebook bootstraps SU2 on a Colab VM and has no off-Colab path.

**Runtime: Colab Pro CPU only.** Do NOT pick GPU (Runtime → Change runtime type → Hardware accelerator: **CPU**). The SU2 benchmark is single-threaded CPU; GPU is wasted quota. PyFR on G4 is the only GPU step in the project and it's Phase 5, not Phase 0.

**What this notebook does:**

1. Clone the `fan-optimization` repo + mount Drive.
2. Install Python deps + SU2 (tries pre-built binary; falls back to source build).
3. Generate the NACA 0012 + probe meshes via Gmsh.
4. Run **Sub-spike 0.6c.1** — Tier-1 cfg sanity (parser-launches-SU2-for-1-step).
5. Run **Sub-spike 0.6c.2** — NACA 0012 oscillating-airfoil benchmark (~6–12 h).
6. Parse SU2 history → per-cycle measured CSV.
7. Run analyzer + aggregator → `data/spike_0_6c/PASS` marker.
8. (Optional) commit + push the PASS marker back to the repo via a GitHub PAT.

**Why this matters.** `scripts/launch_phase4.py` refuses to create the `phase4-launch` git tag without `data/spike_0_6c/PASS`. This is the single non-negotiable hardware-free spike kept in V1 — everything else physical-measurement is deferred to V2 per `docs/phase_logs/phase_0_signoff.md`.

**Runtime expectations:**

| Stage | First run | Subsequent runs (Drive cached) |
|---|---|---|
| Repo clone + Python deps | ~2 min | ~30 s |
| SU2 install (binary path) | ~5 min | ~10 s |
| SU2 install (source build fallback) | ~45 min | ~10 s |
| Mesh generation | ~1 min | ~10 s |
| 0.6c.1 cfg sanity | ~2 min | ~2 min |
| 0.6c.2 SU2 benchmark | **~6–12 h** | ~6–12 h |
| Parse + analyze | ~10 s | ~10 s |

**If Colab disconnects mid-benchmark:** restart the runtime, re-run cells 1–5 (cached), then jump to cell 6. SU2's restart-file mechanism picks up from the last checkpoint.

## Cell 1 — Imports + constants

All imports for the notebook live in this single cell, at the top, unconditionally. Re-run this cell whenever you restart the kernel.

The `google.colab` imports are mandatory — this notebook is Colab-only by design.

In [ ]:
import json
import math
import os
import shutil
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

from google.colab import drive as colab_drive
from google.colab import userdata as colab_userdata

GIT_REPO    = 'https://github.com/clingergab/fan-optimization.git'   # <-- EDIT ME
GIT_BRANCH  = 'main'
GIT_USER    = 'clingergab'    # <-- EDIT ME for cell 9 (PAT push)
GIT_EMAIL   = 'clinger.gab@gmail.com'   # <-- EDIT ME for cell 9
REPO_DIR    = Path('/content/fan-optimization')
DRIVE_DIR   = Path('/content/drive/MyDrive/fan-optimization')
SU2_VERSION = '8.0.1'
SU2_DIR     = Path('/content/su2')
SU2_BIN     = SU2_DIR / 'bin' / 'SU2_CFD'

print(f'[setup] REPO_DIR = {REPO_DIR}; SU2_VERSION = {SU2_VERSION}')

## Cell 2 — Clone repo + mount Drive

In [ ]:
colab_drive.mount('/content/drive', force_remount=False)
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
print(f'[setup] Drive mounted; cache dir: {DRIVE_DIR}')

if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', '--all', '--prune'], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', GIT_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
    print(f'[setup] repo updated at {REPO_DIR}')
else:
    subprocess.run(['git', 'clone', '--branch', GIT_BRANCH, GIT_REPO, str(REPO_DIR)], check=True)
    print(f'[setup] repo cloned to {REPO_DIR}')

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR / 'src'))
sys.path.insert(0, str(REPO_DIR / 'scripts'))
print(f'[setup] cwd = {os.getcwd()}')

## Cell 3 — Install Python dependencies

In [ ]:
# Native libs gmsh dlopens at import time (libGLU + X libs). Colab Pro
# CPU runtimes don't have these by default; without them `import gmsh`
# raises OSError: libGLU.so.1: cannot open shared object file.
!apt-get install -qq -y libglu1-mesa libxrender1 libxcursor1 libxft2 libxinerama1

!pip install --quiet -e {REPO_DIR}
!pip install --quiet gmsh numpy scipy pyyaml jinja2 jsonschema pytest matplotlib pandas
print('[deps] Python deps + gmsh native libs installed')

## Cell 4 — Import fanopt + verify

Cell 3 just installed the `fanopt` package. This cell imports it for the rest of the notebook. Imports are at the top of the cell (per the project rule).

In [ ]:
import fanopt
from fanopt.cfd.configs import render_benchmark_cfg, render_unsteady_cfg
from fanopt.cfd.spike_0_6c import MACH_UNSTEADY_LOCK

print(
    f'[deps] fanopt OK; Tier-1 unsteady MACH lock = {MACH_UNSTEADY_LOCK} '
    f'(Round-9 HIGH-12 / C12)'
)
print(
    '[deps] Sub-spike 0.6c.2 (NACA 0012 benchmark) is DEFERRED-TO-PHASE-5 '
    '(2026-05-XX) -- Cells 8-10 are wind-tunnel-frame prep, NOT a V1 gate.'
)


## Cell 5 — Install SU2

Tries paths in order:
1. **Drive cache** — if a prior run cached `SU2_CFD` on Drive, copy it back.
2. **Pre-built binary** from a recent SU2 GitHub release (Linux x86_64).
3. **Source build** — `meson` + `ninja`. Slow (~30–45 min) but reliable.

After install, the binary is cached to Drive.

In [ ]:
SU2_CACHE = DRIVE_DIR / 'su2_cache' / SU2_VERSION


def _su2_works() -> bool:
    if not SU2_BIN.exists():
        return False
    r = subprocess.run([str(SU2_BIN), '--help'], capture_output=True, text=True)
    return r.returncode in (0, 1)  # SU2 --help can exit 1


def _restore_exec_bits() -> None:
    """Google Drive doesn't preserve POSIX permissions on round-trip — files
    copied back from Drive land at 0o644. Walk SU2_DIR/bin/ and re-set
    +x on every regular file so SU2_CFD (and its companion binaries) are
    invokable. Also re-set +x on any *.so libraries gmsh / SU2 ship with."""
    bin_dir = SU2_DIR / 'bin'
    if bin_dir.is_dir():
        for p in bin_dir.iterdir():
            if p.is_file():
                os.chmod(p, 0o755)
    for p in SU2_DIR.rglob('*.so'):
        if p.is_file():
            os.chmod(p, 0o755)


def _try_drive_cache() -> bool:
    if not SU2_CACHE.exists():
        return False
    SU2_DIR.mkdir(parents=True, exist_ok=True)
    shutil.copytree(SU2_CACHE, SU2_DIR, dirs_exist_ok=True)
    _restore_exec_bits()
    print(f'[su2] restored from Drive cache: {SU2_CACHE} -> {SU2_DIR}')
    return _su2_works()


def _try_binary_release() -> bool:
    url = (
        f'https://github.com/su2code/SU2/releases/download/'
        f'v{SU2_VERSION}/SU2-v{SU2_VERSION}-linux64.zip'
    )
    archive = Path('/tmp/su2.zip')
    try:
        urllib.request.urlretrieve(url, archive)
    except Exception as e:
        print(f'[su2] binary download failed: {e}')
        return False
    SU2_DIR.mkdir(parents=True, exist_ok=True)
    subprocess.run(['unzip', '-q', '-o', str(archive), '-d', str(SU2_DIR)], check=False)
    # Find the real SU2_CFD binary inside the extracted tree. Skip symlinks
    # (a prior run may have left one pointing at the canonical location)
    # and detect the case where the binary is already AT the canonical path
    # (zips that lay bin/SU2_CFD at the top level) so we don't symlink it
    # to itself and trigger ELOOP on the next chmod.
    canonical_path = SU2_DIR / 'bin' / 'SU2_CFD'
    real_bin = None
    for cand in SU2_DIR.rglob('SU2_CFD'):
        if cand.is_symlink() or not cand.is_file():
            continue
        real_bin = cand
        break
    if real_bin is None:
        print('[su2] no SU2_CFD binary found in extracted archive')
        return False
    os.chmod(real_bin, 0o755)
    real_resolved = real_bin.resolve()
    canonical_resolved = canonical_path.resolve() if canonical_path.exists() else None
    # If the real binary is already at the canonical location, no symlink needed.
    if canonical_resolved is not None and real_resolved == canonical_resolved:
        return _su2_works()
    (SU2_DIR / 'bin').mkdir(exist_ok=True)
    if canonical_path.exists() or canonical_path.is_symlink():
        canonical_path.unlink()
    canonical_path.symlink_to(real_bin)
    return _su2_works()


def _try_source_build() -> bool:
    print('[su2] falling back to source build (~30–45 min)...')
    subprocess.run(['apt-get', '-qq', 'install', '-y', 'libopenblas-dev', 'libmpich-dev'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'meson', 'ninja'], check=True)
    src = Path('/tmp/SU2-src')
    if not src.exists():
        subprocess.run(
            ['git', 'clone', '--depth=1', '--branch', f'v{SU2_VERSION}',
             'https://github.com/su2code/SU2.git', str(src)],
            check=True,
        )
    subprocess.run(
        [sys.executable, './meson.py', 'build',
         f'-Dprefix={SU2_DIR}', '-Denable-autodiff=false'],
        cwd=src, check=True,
    )
    subprocess.run(['ninja', '-C', 'build', 'install'], cwd=src, check=True)
    return _su2_works()


def _cache_to_drive() -> None:
    if SU2_CACHE.exists():
        return
    print(f'[su2] caching binary to {SU2_CACHE} for future runs...')
    SU2_CACHE.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(SU2_DIR, SU2_CACHE)


installed = _try_drive_cache() or _try_binary_release() or _try_source_build()
if not installed:
    raise RuntimeError(
        'SU2 install failed via all paths. Inspect logs above; common fixes: '
        'bump SU2_VERSION to the latest release; check the GitHub asset name; '
        'try the source build directly.'
    )
_cache_to_drive()
os.environ['PATH'] = f'{SU2_DIR / "bin"}:{os.environ["PATH"]}'
print(f'[su2] SU2_CFD on PATH at {SU2_BIN}')
!SU2_CFD --help | head -5

## Cell 6 — Generate benchmark meshes

Builds the probe mesh (0.6c.1) and NACA 0012 O-grid mesh (0.6c.2). Mesh files go to `data/spike_0_6c/meshes/` and cache on Drive.

In [ ]:
MESH_DIR = REPO_DIR / 'data' / 'spike_0_6c' / 'meshes'
DRIVE_MESH_DIR = DRIVE_DIR / 'spike_0_6c' / 'meshes'
MESH_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_MESH_DIR.mkdir(parents=True, exist_ok=True)

# `python3 -u` disables stdout buffering so the `--verbose` progress
# markers stream into the notebook in real time instead of dumping
# all at once when the process exits.
probe_mesh = MESH_DIR / 'probe.su2'
if not probe_mesh.exists() and (DRIVE_MESH_DIR / 'probe.su2').exists():
    shutil.copy(DRIVE_MESH_DIR / 'probe.su2', probe_mesh)
if not probe_mesh.exists():
    !python3 -u scripts/gen_benchmark_meshes.py --kind probe --out {probe_mesh}
    shutil.copy(probe_mesh, DRIVE_MESH_DIR / 'probe.su2')

naca_mesh = MESH_DIR / 'naca0012.su2'
if not naca_mesh.exists() and (DRIVE_MESH_DIR / 'naca0012.su2').exists():
    shutil.copy(DRIVE_MESH_DIR / 'naca0012.su2', naca_mesh)
if not naca_mesh.exists():
    !python3 -u scripts/gen_benchmark_meshes.py --kind naca0012 --verbose --out {naca_mesh}
    shutil.copy(naca_mesh, DRIVE_MESH_DIR / 'naca0012.su2')

print(f'[meshes] probe:     {probe_mesh.stat().st_size / 1024:.1f} KB')
print(f'[meshes] naca0012:  {naca_mesh.stat().st_size / 1024:.1f} KB')

## Cell 7 — Sub-spike 0.6c.1 (cfg sanity)

Renders the production Tier-1 unsteady cfg, runs SU2 for ≥1 outer iter on the probe mesh, asserts the Round-9 HIGH-12 / C11 invariants. **Pass criterion:** SU2 launches + completes 1 outer time-step without parser error. ~2 min.

In [ ]:
# Sub-spike 0.6c.1 — cfg sanity + outer-step evidence
# Post-2026-05-14: the runner accepts --su2-history-csv as evidence from a
# prior successful SU2 run (e.g., the Cell 8 NACA 0012 run already on Drive).
# This avoids needing to re-invoke SU2 for the cfg-sanity check — the existing
# history.csv from May 2026 IS proof SU2 accepts the Tier-1 numerics, since
# the benchmark cfg uses identical MACH=1e-9 + REF_DIMENSIONALIZATION numerics.

result_dir = REPO_DIR / 'data' / 'spike_0_6c' / 'sub_1_run'
result_dir.mkdir(parents=True, exist_ok=True)

# Prefer evidence from the existing Cell 8 history.csv on Drive (no re-run needed).
prior_history = DRIVE_DIR / 'spike_0_6c' / 'sub_2_run' / 'history.csv'

if prior_history.exists():
    print(f'[0.6c.1] using prior-run history.csv as evidence: {prior_history}')
    !python3 scripts/run_spike_0_6c_1.py \
        --su2-history-csv {prior_history} \
        --result-json {REPO_DIR}/data/spike_0_6c/sub_1_result.json \
        --marker-dir {REPO_DIR}/data/spike_0_6c
else:
    print(f'[0.6c.1] no prior history.csv at {prior_history}; falling back to fresh probe')
    probe_mesh = REPO_DIR / 'data' / 'spike_0_6c' / 'meshes' / 'probe.su2'
    if not probe_mesh.exists():
        !python3 scripts/gen_benchmark_meshes.py --kind probe --out {probe_mesh}
    !python3 scripts/run_spike_0_6c_1.py \
        --probe-mesh {probe_mesh} \
        --probe-workdir {result_dir} \
        --result-json {REPO_DIR}/data/spike_0_6c/sub_1_result.json \
        --marker-dir {REPO_DIR}/data/spike_0_6c


## Cell 8 — Sub-spike 0.6c.2 (NACA 0012 benchmark) — **DEFERRED TO PHASE 5 (2026-05-XX)**

**Do not run for V1.** Sub-spike 0.6c.2 was deferred to Phase 5 on 2026-05-XX after the regime diagnostic on this cell's previous output confirmed the production-faithful MACH=1e-9 cfg produces body-in-still-air added-mass/quadratic-drag forces — NOT wind-tunnel-like aerodynamic lift — and therefore can't be validated against any published wind-tunnel NACA 0012 oscillating-airfoil benchmark in the same frame.

Full decision record: `docs/phase_logs/spike_0_6c.md` → "V1 decision — Spike 0.6c.2 deferred to Phase 5 (2026-05-XX)".

**Phase 5 deliverable** (the new home of 0.6c.2): SU2 vs PyFR cross-solver agreement on a published wind-tunnel NACA 0012 case in the conventional frame. Acceptance: C_L_max within ±20%, C_d_mean within ±25%, hysteresis loop sign matches + area within 2×.

**For V1:** stop here. Cells 9 and 10 below are also DEFERRED-TO-PHASE-5 and should not be run. To unblock the Phase 4 launch tag, only Cell 7 (sub-spike 0.6c.1 cfg sanity) needs to produce `sub_1.PASS` — the aggregator (`run_spike_0_6c.py`) then writes `data/spike_0_6c/PASS` which is what `launch_phase4.py` gates on.

If you want to re-validate the diagnostic finding yourself, the `scripts/diagnose_su2_pitching_regime.py` script (in the next cell on a fresh notebook run) reads the existing history.csv and reports the regime classification + bias.

In [ ]:
# --- WIND-TUNNEL-FRAME NACA 0012 PITCHING BENCHMARK (Phase-5 prep) --------
#
# DEFERRED-TO-PHASE-5 on 2026-05-XX. Do not run for V1 sign-off.
#
# Numerics rationale for whoever picks this up in Phase 5:
#   - Strategy A (this cell, default): pin V via Re on a 1m chord -> MACH
#     drops to ~1.7e-3. Stays close to incompressible; matches the
#     k_reduced exactly but pushes the low-Mach preconditioner hard.
#   - Strategy B: shrink chord to ~0.03 m so MACH=0.05 -> V~17 m/s gives
#     Re~40000 and a conventional compressible-solver regime. Requires
#     regenerating the NACA 0012 mesh at the smaller chord. Cleaner
#     numerics, but needs `scripts/gen_benchmark_meshes.py --chord 0.03`.
# Phase 5 should pick one consciously; the wind-tunnel-frame cfg
# template supports both (just pass `mach_number=...` to
# render_benchmark_cfg).

REYNOLDS_NUMBER  = 40000.0
REYNOLDS_LENGTH  = 1.0  # chord (m) -- Strategy A
K_REDUCED        = 0.55
KINEMATIC_VISC_M2_PER_S = 1.5e-5  # sea-level standard air
V_FREESTREAM     = REYNOLDS_NUMBER * KINEMATIC_VISC_M2_PER_S / REYNOLDS_LENGTH
F_PITCH_HZ       = K_REDUCED * V_FREESTREAM / (math.pi * REYNOLDS_LENGTH)
OMEGA_PITCH      = 2.0 * math.pi * F_PITCH_HZ
AMPL_PITCH_RAD   = math.radians(10.0)
N_CYCLES         = 5
T_PERIOD         = 2.0 * math.pi / OMEGA_PITCH
DT               = T_PERIOD / 200.0
TIME_ITER        = N_CYCLES * 200

# MACH from V_FREESTREAM / c_inf at T_inf = 300 K (Strategy A).
SOUND_SPEED_INF  = math.sqrt(1.4 * 287.05 * 300.0)
MACH_NUMBER      = V_FREESTREAM / SOUND_SPEED_INF

bench_cfg_text = render_benchmark_cfg(
    mesh_filename=str(naca_mesh),
    marker_airfoil='AIRFOIL',
    marker_farfield='FARFIELD',
    mach_number=MACH_NUMBER,
    reynolds_number=REYNOLDS_NUMBER,
    reynolds_length=REYNOLDS_LENGTH,
    pitching_omega_y=OMEGA_PITCH,
    pitching_ampl_y=AMPL_PITCH_RAD,
    motion_origin_x=0.25,
    time_step=DT,
    max_time=N_CYCLES * T_PERIOD,
    time_iter=TIME_ITER,
    inner_iter=80,
)

bench_dir = DRIVE_DIR / 'spike_0_6c' / 'phase5_prep_benchmark'
bench_dir.mkdir(parents=True, exist_ok=True)
bench_cfg = bench_dir / 'naca0012_pitch.cfg'
bench_cfg.write_text(bench_cfg_text)
print(f'[phase5-prep] cfg -> {bench_cfg}')
print(f'[phase5-prep] V={V_FREESTREAM:.4f} m/s  MACH={MACH_NUMBER:.5f}  '
      f'omega={OMEGA_PITCH:.4f} rad/s  T={T_PERIOD:.4f} s  dt={DT:.4e} s')
print(f'[phase5-prep] TIME_ITER={TIME_ITER}; expected wall-time ~6-12 h on Colab Pro CPU')

log_path = bench_dir / 'su2.log'
with log_path.open('w') as logf:
    proc = subprocess.Popen(
        ['SU2_CFD', str(bench_cfg)],
        cwd=bench_dir,
        stdout=logf, stderr=subprocess.STDOUT,
    )
    while proc.poll() is None:
        time.sleep(60)
        tail = subprocess.run(
            ['tail', '-1', str(log_path)], capture_output=True, text=True
        ).stdout.strip()
        print(f'[phase5-prep heartbeat] {tail[:160]}')

rc = proc.returncode
print(f'[phase5-prep] SU2 exit={rc}')
if rc != 0:
    print('---LOG TAIL---'); print('\n'.join(log_path.read_text().splitlines()[-40:]))


## Cell 9 — Parse SU2 history → measured.csv — **DEFERRED-TO-PHASE-5**

Cell skipped in V1. The downstream analyzer (`run_spike_0_6c_2.py`) was removed when 0.6c.2 was deferred to Phase 5; running this cell produces a `measured.csv` that has no V1 consumer. The `parse_su2_history_to_cycles.py` script remains in the repo for Phase 5 cross-solver use.

In [ ]:
history_csv = bench_dir / 'history.csv'
if not history_csv.exists():
    candidates = list(bench_dir.glob('history*.csv')) + list(bench_dir.glob('history*.dat'))
    if not candidates:
        raise FileNotFoundError(f'No SU2 history file in {bench_dir}. SU2 may have failed; check log.')
    history_csv = candidates[0]
    print(f'[parse] using {history_csv.name}')

measured_csv = bench_dir / 'measured.csv'
!python3 scripts/parse_su2_history_to_cycles.py \
    --history {history_csv} \
    --n-cycles {N_CYCLES} \
    --omega-shm-rad-per-s {OMEGA_PITCH} \
    --theta-max-rad {AMPL_PITCH_RAD} \
    --out {measured_csv}

print(measured_csv.read_text())

## Cell 10 — Run analyzer + aggregator — **PARTIALLY DEFERRED-TO-PHASE-5**

The analyzer (`run_spike_0_6c_2.py`) was removed when 0.6c.2 was deferred to Phase 5 — that part of the cell will fail with `ModuleNotFoundError`. The aggregator (`run_spike_0_6c.py`) still runs and is REQUIRED to write the `data/spike_0_6c/PASS` marker that `launch_phase4.py` gates on.

**V1 workflow:** after Cell 7 writes `sub_1.PASS`, run the aggregator + launch_phase4 cells below in isolation:

```python
spike_0_6c_dir = REPO_DIR / 'data' / 'spike_0_6c'
spike_0_6c_dir.mkdir(parents=True, exist_ok=True)
!python3 scripts/run_spike_0_6c.py \
    --sub-1-json {result_dir}/sub_1_result.json \
    --out {spike_0_6c_dir}/results.json \
    --marker-dir {spike_0_6c_dir}
!python3 scripts/launch_phase4.py --check
```

The aggregator now reads ONLY the sub_1 result. The `--sub-2-json` flag and the analyzer step are gone with the 0.6c.2 deferral.

In [ ]:
spike_0_6c_dir = REPO_DIR / "data" / "spike_0_6c"
spike_0_6c_dir.mkdir(parents=True, exist_ok=True)

# V1 aggregator (post-2026-05-XX): reads ONLY sub_1. The 0.6c.2
# analyzer (run_spike_0_6c_2.py) was removed when 0.6c.2 was deferred
# to Phase 5 — see docs/phase_logs/spike_0_6c.md.
!python3 scripts/run_spike_0_6c.py \
    --sub-1-json {result_dir}/sub_1_result.json \
    --out {spike_0_6c_dir}/results.json \
    --marker-dir {spike_0_6c_dir}

!python3 scripts/launch_phase4.py --check

result = json.loads((spike_0_6c_dir / "results.json").read_text())
print("\n=== FINAL ===")
print(f"overall_passed: {result['result']['overall_passed']}")
print(f"sub_0.6c.1:    {result['result']['sub_06c_1']['passed']}")
print("sub_0.6c.2:    DEFERRED-TO-PHASE-5 (2026-05-XX)")

## Cell 11 — (Optional) commit + push PASS marker

Only run this if you want the marker committed back. Requires a GitHub PAT with `repo` scope set as the Colab secret `GITHUB_PAT` (left sidebar → key icon → Add new secret).

If the secret isn't set, `colab_userdata.get('GITHUB_PAT')` raises; the cell prints a skip notice and exits cleanly.

In [ ]:
PAT = None
try:
    PAT = colab_userdata.get('GITHUB_PAT')
except Exception as e:
    print(f'[commit] GITHUB_PAT secret not configured ({e}); skipping push.')

if PAT:
    subprocess.run(['git', 'config', '--global', 'user.name', GIT_USER], check=True)
    subprocess.run(['git', 'config', '--global', 'user.email', GIT_EMAIL], check=True)
    push_url = GIT_REPO.replace('https://', f'https://{GIT_USER}:{PAT}@')
    !git -C {REPO_DIR} add data/spike_0_6c/PASS data/spike_0_6c/results.json
    !git -C {REPO_DIR} commit -m 'Spike 0.6c: PASS marker from Colab run' -m 'Auto-committed from notebook'
    !git -C {REPO_DIR} push {push_url} {GIT_BRANCH}
    print('[commit] PASS marker pushed.')

## Done.

If both sub-spikes passed, `data/spike_0_6c/PASS` is in the repo and `scripts/launch_phase4.py` will accept the `phase4-launch` tag.

If anything failed: see `notebooks/README.md` → "Troubleshooting" for the common cases.